# **Datasets**

## **Lectura y revisión**

In [ ]:
import pandas as pd

dataset_entrenamiento = pd.read_csv('../data/train_clean.csv')
dataset_prueba        = pd.read_csv('../data/test_clean.csv')
dataset_oot           = pd.read_csv('../data/oot_clean.csv')

In [ ]:
dataset_entrenamiento.info()

## **X & Y**

In [ ]:
# entrenamiento
X_entrenamiento = dataset_entrenamiento.drop(columns=['es_fraude'])
Y_entrenamiento = dataset_entrenamiento['es_fraude']

# prueba
X_prueba        = dataset_prueba.drop(columns = ['es_fraude'])
Y_prueba        = dataset_prueba['es_fraude']

# fuera de tiempo
X_oot           = dataset_oot.drop(columns =['es_fraude'])
Y_oot           = dataset_oot['es_fraude']

---
# **Hiperparámetros**

In [ ]:
ceros_entrena  = (Y_entrenamiento == 0).sum()
unos_entrena   = (Y_entrenamiento == 1).sum()

peso_positivos = ceros_entrena/unos_entrena

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna


def objetivo(intento):
    params = {
        'num_leaves': intento.suggest_int('num_leaves', 15, 255),
        'max_depth': intento.suggest_int('max_depth', 3, 12),
        'min_child_samples': intento.suggest_int('min_child_samples', 5, 100),
        'learning_rate': intento.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': intento.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq': 1,  # necesario para que 'subsample' realmente se aplique
        'colsample_bytree': intento.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': intento.suggest_float('reg_alpha', 1e-3, 5.0, log=True),
        'reg_lambda': intento.suggest_float('reg_lambda', 1e-2, 5.0, log=True),
        'min_split_gain': intento.suggest_float('min_split_gain', 0, 5),
        'scale_pos_weight': peso_positivos,
        'n_estimators': 500,
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': -1,
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    auc_scores = []

    for train_idx, val_idx in cv.split(X_entrenamiento, Y_entrenamiento):
        X_tr, X_val = X_entrenamiento.iloc[train_idx], X_entrenamiento.iloc[val_idx]
        y_tr, y_val = Y_entrenamiento.iloc[train_idx], Y_entrenamiento.iloc[val_idx]

        modelo = LGBMClassifier(**params)
        modelo.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric='auc',
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
        )

        preds = modelo.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, preds))

    return np.mean(auc_scores)



study = optuna.create_study(direction='maximize')
study.optimize(objetivo, n_trials=30)
print("Mejores hiperparámetros:", study.best_params)
print("Mejor AUC:", study.best_value)

**Los mejores hiperparámetros fueron:**

num_leaves: 212
Cerca del techo (rango 15-255). Es el hiperparámetro más importante en LightGBM por su crecimiento leaf-wise: el modelo pidió árboles con muchas hojas para capturar patrones finos entre fraude y no-fraude.

max_depth: 5
Bajo en comparación con num_leaves alto. Actúa más como techo de seguridad que como controlador principal — LightGBM prioriza splits de mayor ganancia sin importar el nivel, en vez de crecer en profundidad.

min_child_samples: 95
Cerca del techo (rango 5-100). Equivalente a min_child_weight en XGBoost: exige suficientes observaciones por hoja, compensando el num_leaves alto y evitando que el modelo memorice casos aislados de fraude.

learning_rate: 0.0487
Bajo-medio dentro del rango 0.01-0.3. Coherente con tener más hojas disponibles: un ritmo más conservador evita sobreajuste rápido, apoyado en los 500 árboles de techo con early stopping.

subsample: 0.807
Cada árbol usa ~81% de las filas. Similar a XGBoost (0.788), confirma que ese nivel de submuestreo reduce varianza en este dataset.

colsample_bytree: 0.961
Muy alto. La mayoría de las variables aportan señal útil, por lo que el modelo casi no necesita descartar columnas.

reg_alpha: 0.0246 / reg_lambda: 0.0273
Ambos cerca del piso (rango 1e-3 a 5.0). Casi sin regularización L1/L2 — el control de sobreajuste recae principalmente en min_child_samples.

min_split_gain: 0.785
Moderado-bajo (rango 0-5). Equivalente a gamma en XGBoost, pero aquí con un rol secundario frente a min_child_samples.


**Mejor AUC**: 0.993876688771147

In [ ]:
mejores_params = {
    'num_leaves': 212,
    'max_depth': 5,
    'min_child_samples': 95,
    'learning_rate': 0.04869327777115842,
    'subsample': 0.8070706812552854,
    'subsample_freq': 1,
    'colsample_bytree': 0.9605756757451125,
    'reg_alpha': 0.02459914729522153,
    'reg_lambda': 0.02729667831232282,
    'min_split_gain': 0.7849721695343561,
    'scale_pos_weight': peso_positivos,
    'n_estimators': 500,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': -1,
}

lightgbm = LGBMClassifier(**mejores_params)
lightgbm.fit(
    X_entrenamiento, Y_entrenamiento,
    eval_set=[(X_prueba, Y_prueba)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

In [ ]:
import numpy as np
from sklearn.metrics import fbeta_score, classification_report, confusion_matrix, roc_auc_score, f1_score

y_proba_prueba_lgb = lightgbm.predict_proba(X_prueba)[:, 1]
mejor_umbral_lgb = 0.5
mejor_f2_lgb = 0.0

for umbral in np.arange(0.01, 1.0, 0.01):
    prediccion_temporal = (y_proba_prueba_lgb >= umbral).astype(int)
    f2_temporal = fbeta_score(Y_prueba, prediccion_temporal, beta=2, zero_division=0)
    if f2_temporal > mejor_f2_lgb:
        mejor_f2_lgb = f2_temporal
        mejor_umbral_lgb = umbral

print(f"Umbral ganador: {mejor_umbral_lgb:.2f}")
print(f"Mejor F2 en prueba: {mejor_f2_lgb:.4f}")

In [ ]:
def evaluar_con_umbral(y_real, y_proba, umbral, nombre_set):
    pred = (y_proba >= umbral).astype(int)
    print(f"\n{'='*50}")
    print(f"Resultados en {nombre_set} (umbral = {umbral:.2f})")
    print(f"{'='*50}")
    print(f"AUC-ROC: {roc_auc_score(y_real, y_proba):.4f}")
    print(f"F2-score: {fbeta_score(y_real, pred, beta=2):.4f}")
    print(f"\n{classification_report(y_real, pred, target_names=['No fraude', 'Fraude'], digits=5)}")
    print("Matriz de confusión:")
    print(confusion_matrix(y_real, pred))

evaluar_con_umbral(Y_prueba, y_proba_prueba_lgb, mejor_umbral_lgb, "Prueba")

y_proba_oot_lgb = lightgbm.predict_proba(X_oot)[:, 1]
evaluar_con_umbral(Y_oot, y_proba_oot_lgb, mejor_umbral_lgb, "OOT")